# Notebook 12: Quantum Walks

**Series: Quantum Computing from Ground Up**

---

## Learning Objectives

1. Review classical random walks and their properties
2. Understand discrete-time quantum walks (DTQW): coin and shift operators
3. Explore continuous-time quantum walks (CTQW)
4. Simulate quantum walks on a line, cycle, and graph
5. Compare quantum and classical walk distributions
6. Applications to search algorithms and graph problems

## 1. Classical Random Walks Review

### 1.1 Random Walk on a Line

A **classical random walk** on $\mathbb{Z}$ starts at position $x=0$. At each time step, the walker moves:
- Left ($x \to x-1$) with probability $1/2$
- Right ($x \to x+1$) with probability $1/2$

After $t$ steps, the probability distribution is **binomial**:

$$P(x, t) = \binom{t}{(t+x)/2} \left(\frac{1}{2}\right)^t$$

(defined for $x$ with same parity as $t$, and $|x| \leq t$).

### 1.2 Key Properties

- **Mean position:** $\langle x \rangle = 0$ (unbiased)
- **Standard deviation:** $\sigma = \sqrt{t}$ (diffusive spreading)
- **Mixing time** on a cycle of $N$ nodes: $O(N^2)$
- **Hitting time** on a line: $O(N^2)$ to reach position $N$

The $\sqrt{t}$ spreading is the hallmark of **diffusion**. Quantum walks will achieve **linear** $\sigma \propto t$ spreading — a quadratic speedup.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb

def classical_random_walk(t_steps, n_walkers=10000):
    """Simulate classical random walk on a line."""
    # Each step is +1 or -1 with equal probability
    steps = np.random.choice([-1, 1], size=(n_walkers, t_steps))
    positions = np.cumsum(steps, axis=1)
    return positions

# Simulate
np.random.seed(42)
t_max = 100
positions = classical_random_walk(t_max, n_walkers=50000)

# Plot distribution at different times
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, t in enumerate([10, 50, 100]):
    ax = axes[idx]
    final_positions = positions[:, t-1]
    
    ax.hist(final_positions, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
    
    # Gaussian approximation
    x = np.linspace(-3*np.sqrt(t), 3*np.sqrt(t), 200)
    gaussian = np.exp(-x**2 / (2*t)) / np.sqrt(2*np.pi*t)
    ax.plot(x, gaussian, 'r-', linewidth=2, label=f'Gaussian σ={np.sqrt(t):.1f}')
    
    ax.set_title(f't = {t} steps', fontsize=13)
    ax.set_xlabel('Position')
    ax.set_ylabel('Probability density')
    ax.legend(fontsize=10)

plt.suptitle('Classical Random Walk on a Line', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Verify sigma ~ sqrt(t)
times = [10, 20, 50, 100]
sigmas = [np.std(positions[:, t-1]) for t in times]
print("\nStandard deviation vs time:")
for t, s in zip(times, sigmas):
    print(f"  t={t:3d}: σ={s:.2f}, √t={np.sqrt(t):.2f}, ratio={s/np.sqrt(t):.3f}")

## 2. Discrete-Time Quantum Walk (DTQW)

### 2.1 Hilbert Space

A discrete-time quantum walk on a line uses two registers:

$$\mathcal{H} = \mathcal{H}_C \otimes \mathcal{H}_P$$

- **Coin space** $\mathcal{H}_C$: 2-dimensional (analogous to the coin flip)
- **Position space** $\mathcal{H}_P$: spans positions $\{|x\rangle : x \in \mathbb{Z}\}$

A general state is:

$$|\psi(t)\rangle = \sum_x \left(\alpha_x(t)|0\rangle_C + \beta_x(t)|1\rangle_C\right) \otimes |x\rangle_P$$

### 2.2 The Coin Operator

The **Hadamard coin** is the most common choice:

$$C_H = H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

Other coin choices:

- **Grover coin** (for higher dimensions): $G = 2|s\rangle\langle s| - I$, where $|s\rangle = \frac{1}{\sqrt{d}}\sum_{i=0}^{d-1}|i\rangle$
- **Balanced coin:** $C_Y = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & i \\ i & 1 \end{pmatrix}$

### 2.3 The Shift Operator

The **conditional shift** moves the walker based on the coin state:

$$S = |0\rangle\langle 0|_C \otimes \sum_x |x-1\rangle\langle x|_P + |1\rangle\langle 1|_C \otimes \sum_x |x+1\rangle\langle x|_P$$

In words: if coin = $|0\rangle$, move left; if coin = $|1\rangle$, move right.

### 2.4 One Step of the Walk

The evolution operator for one step is:

$$U = S \cdot (C \otimes I_P)$$

After $t$ steps:

$$|\psi(t)\rangle = U^t |\psi(0)\rangle$$

In [ ]:
def discrete_quantum_walk_1d(t_steps, coin='hadamard', initial_coin='zero'):
    """
    Simulate a discrete-time quantum walk on a 1D line.
    
    Parameters:
        t_steps: number of time steps
        coin: 'hadamard' or 'balanced'
        initial_coin: 'zero', 'one', 'plus', or 'minus'
    
    Returns:
        positions: array of position values
        probabilities: probability at each position
    """
    N = 2 * t_steps + 1  # Total positions needed
    
    # State vector: [coin_0_pos_0, coin_0_pos_1, ..., coin_1_pos_0, coin_1_pos_1, ...]
    # Shape: (2, N) where 2 is coin dimension, N is position dimension
    state = np.zeros((2, N), dtype=complex)
    
    # Initial state at position t_steps (center)
    center = t_steps
    if initial_coin == 'zero':
        state[0, center] = 1.0
    elif initial_coin == 'one':
        state[1, center] = 1.0
    elif initial_coin == 'plus':
        state[0, center] = 1.0 / np.sqrt(2)
        state[1, center] = 1.0 / np.sqrt(2)
    elif initial_coin == 'minus':
        state[0, center] = 1.0 / np.sqrt(2)
        state[1, center] = -1.0 / np.sqrt(2)
    
    # Coin matrices
    if coin == 'hadamard':
        C = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    elif coin == 'balanced':
        C = np.array([[1, 1j], [1j, 1]]) / np.sqrt(2)
    
    # Evolution
    for step in range(t_steps):
        # Apply coin operator
        new_state = np.zeros_like(state)
        for x in range(N):
            coin_state = state[:, x]
            new_coin = C @ coin_state
            new_state[:, x] = new_coin
        
        # Apply shift operator
        shifted_state = np.zeros_like(state)
        for x in range(N):
            # Coin 0 -> move left
            if x > 0:
                shifted_state[0, x-1] += new_state[0, x]
            # Coin 1 -> move right
            if x < N-1:
                shifted_state[1, x+1] += new_state[1, x]
        
        state = shifted_state
    
    # Compute probabilities
    probabilities = np.abs(state[0])**2 + np.abs(state[1])**2
    positions = np.arange(N) - center
    
    return positions, probabilities

# Simulate quantum walk
t = 50
pos_qw, prob_qw = discrete_quantum_walk_1d(t, coin='hadamard', initial_coin='zero')

print(f"Quantum walk after {t} steps:")
print(f"  Total probability: {np.sum(prob_qw):.6f}")
print(f"  Mean position: {np.sum(pos_qw * prob_qw):.4f}")
print(f"  Std deviation: {np.sqrt(np.sum(pos_qw**2 * prob_qw) - np.sum(pos_qw * prob_qw)**2):.4f}")
print(f"  Expected σ ~ t/√2 = {t/np.sqrt(2):.4f}")

In [ ]:
# Compare quantum walk vs classical walk distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, t in enumerate([20, 50, 100]):
    ax = axes[idx]
    
    # Quantum walk
    pos_qw, prob_qw = discrete_quantum_walk_1d(t, coin='hadamard', initial_coin='zero')
    ax.bar(pos_qw, prob_qw, width=1.0, alpha=0.6, color='blue', label='Quantum walk')
    
    # Classical walk (binomial)
    x_vals = np.arange(-t, t+1)
    prob_classical = np.zeros_like(x_vals, dtype=float)
    for i, x in enumerate(x_vals):
        if (t + x) % 2 == 0 and abs(x) <= t:
            k = (t + x) // 2
            prob_classical[i] = comb(t, k, exact=True) * (0.5)**t
    
    ax.plot(x_vals, prob_classical, 'r-', linewidth=2, label='Classical walk')
    
    ax.set_title(f't = {t} steps', fontsize=13)
    ax.set_xlabel('Position', fontsize=11)
    ax.set_ylabel('Probability', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_xlim(-t, t)

plt.suptitle('Quantum Walk vs Classical Random Walk on a Line', fontsize=14)
plt.tight_layout()
plt.show()

print("Key differences:")
print("  1. Quantum walk spreads LINEARLY (σ ~ t), classical spreads as √t")
print("  2. Quantum walk has interference peaks at the edges")
print("  3. Quantum walk is ASYMMETRIC with Hadamard coin + |0> initial coin")

## 3. Effect of Initial Coin State

The asymmetry in the Hadamard walk depends on the initial coin state. A symmetric distribution is obtained with $|\psi_C\rangle = \frac{1}{\sqrt{2}}(|0\rangle - i|1\rangle)$, or equivalently by using the **balanced coin** $C_Y$.

Mathematically, the Hadamard coin treats $|0\rangle$ and $|1\rangle$ asymmetrically:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \quad H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

The minus sign in $H|1\rangle$ causes destructive interference on the left side when starting from $|0\rangle_C$.

In [ ]:
# Effect of initial coin state
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
t = 50

configs = [
    ('hadamard', 'zero', 'Hadamard coin, |0⟩ initial'),
    ('hadamard', 'one', 'Hadamard coin, |1⟩ initial'),
    ('hadamard', 'plus', 'Hadamard coin, |+⟩ initial'),
    ('balanced', 'zero', 'Balanced coin (Y), |0⟩ initial'),
]

for idx, (coin, init, title) in enumerate(configs):
    ax = axes[idx // 2][idx % 2]
    pos, prob = discrete_quantum_walk_1d(t, coin=coin, initial_coin=init)
    
    ax.bar(pos, prob, width=1.0, alpha=0.7, color='steelblue')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Position')
    ax.set_ylabel('Probability')
    ax.set_xlim(-t, t)
    
    sigma = np.sqrt(np.sum(pos**2 * prob) - np.sum(pos * prob)**2)
    ax.text(0.02, 0.95, f'σ = {sigma:.2f}', transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Quantum Walk on a Line (t={t} steps): Effect of Coin and Initial State', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Continuous-Time Quantum Walk (CTQW)

### 4.1 Definition

A **continuous-time quantum walk** on a graph $G = (V, E)$ evolves according to the Schrodinger equation with the graph's adjacency matrix (or Laplacian) as the Hamiltonian:

$$|\psi(t)\rangle = e^{-iAt}|\psi(0)\rangle$$

where $A$ is the adjacency matrix of the graph. No coin is needed — the graph structure itself drives the walk.

### 4.2 Evolution on a Line

For a path graph of $N$ nodes, the adjacency matrix is tridiagonal:

$$A = \begin{pmatrix}
0 & 1 & & \\
1 & 0 & 1 & \\
  & 1 & 0 & \ddots \\
  &   & \ddots & \ddots
\end{pmatrix}$$

The eigenstates are standing waves, and the evolution is:

$$\langle x | \psi(t) \rangle = \sum_k c_k e^{-i\lambda_k t} \phi_k(x)$$

where $\lambda_k$ are eigenvalues and $\phi_k$ are eigenvectors of $A$.

### 4.3 Comparison: DTQW vs CTQW

| Property | DTQW | CTQW |
|----------|------|------|
| Time | Discrete steps | Continuous |
| Coin | Required | Not needed |
| Hilbert space | Coin ⊗ Position | Position only |
| Universality | Yes | Yes |
| Spreading | $\sigma \propto t$ | $\sigma \propto t$ |

In [ ]:
from scipy.linalg import expm

def continuous_quantum_walk(adjacency, initial_node, times):
    """
    Simulate continuous-time quantum walk on a graph.
    
    Parameters:
        adjacency: NxN adjacency matrix
        initial_node: starting node index
        times: list of time points
    
    Returns:
        probabilities: dict mapping time -> probability array
    """
    N = adjacency.shape[0]
    psi0 = np.zeros(N, dtype=complex)
    psi0[initial_node] = 1.0
    
    probabilities = {}
    for t in times:
        U = expm(-1j * adjacency * t)
        psi_t = U @ psi0
        probabilities[t] = np.abs(psi_t)**2
    
    return probabilities

# CTQW on a line graph
N = 51
A_line = np.zeros((N, N))
for i in range(N-1):
    A_line[i, i+1] = 1
    A_line[i+1, i] = 1

times = [1, 5, 10, 15, 20, 25]
probs = continuous_quantum_walk(A_line, N//2, times)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
positions = np.arange(N) - N//2

for idx, t in enumerate(times):
    ax = axes[idx // 3][idx % 3]
    ax.bar(positions, probs[t], width=1.0, alpha=0.7, color='darkgreen')
    ax.set_title(f't = {t}', fontsize=12)
    ax.set_xlabel('Position')
    ax.set_ylabel('Probability')
    ax.set_xlim(-N//2, N//2)
    ax.set_ylim(0, max(0.3, max(probs[t]) * 1.1))

plt.suptitle('Continuous-Time Quantum Walk on a Line', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Quantum Walk on a Cycle

### 5.1 CTQW on a Cycle

A cycle graph $C_N$ has adjacency matrix with additional wraparound connections. The eigenvalues of the cycle adjacency matrix are:

$$\lambda_k = 2\cos\left(\frac{2\pi k}{N}\right), \quad k = 0, 1, \ldots, N-1$$

with eigenvectors $|\phi_k\rangle$ where $\langle j | \phi_k \rangle = \frac{1}{\sqrt{N}} e^{2\pi i jk/N}$ (Fourier modes).

### 5.2 Mixing on a Cycle

The **mixing time** is the time to reach a nearly uniform distribution. Classically, the mixing time on $C_N$ is $O(N^2)$. Quantum walks achieve $O(N \log N)$ mixing in the **time-averaged** sense:

$$\bar{P}(x, T) = \frac{1}{T} \int_0^T |\langle x | e^{-iAt} | 0 \rangle|^2 \, dt$$

In [ ]:
# CTQW on a cycle graph
N = 21
A_cycle = np.zeros((N, N))
for i in range(N):
    A_cycle[i, (i+1) % N] = 1
    A_cycle[(i+1) % N, i] = 1

times_cycle = [0, 2, 5, 8, 10, 15]
probs_cycle = continuous_quantum_walk(A_cycle, 0, times_cycle)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
nodes = np.arange(N)

for idx, t in enumerate(times_cycle):
    ax = axes[idx // 3][idx % 3]
    ax.bar(nodes, probs_cycle[t], width=0.8, alpha=0.7, color='purple')
    ax.axhline(y=1/N, color='red', linestyle='--', label=f'Uniform = {1/N:.3f}')
    ax.set_title(f't = {t}', fontsize=12)
    ax.set_xlabel('Node')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)

plt.suptitle(f'CTQW on Cycle C_{N} (starting from node 0)', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: CTQW doesn't converge to uniform in instantaneous distribution.")
print("Time-averaged distribution converges more uniformly.")

## 6. Quantum Walk on General Graphs

### 6.1 Graph Properties and Walk Behavior

The structure of the graph profoundly affects the quantum walk. The **spectral gap** $\Delta = \lambda_1 - \lambda_2$ (difference between largest and second-largest eigenvalues) determines mixing properties.

### 6.2 Quantum Walk on a Hypercube

The $n$-dimensional hypercube $Q_n$ has $2^n$ nodes. Quantum walks on hypercubes exhibit:
- **Exponential speedup** for hitting: classical $O(2^n)$ vs quantum $O(n)$
- This is related to Grover's search speedup

In [ ]:
# Quantum walk on various graph structures
import networkx as nx

def plot_graph_walk(G, name, start_node=0):
    """Simulate and visualize CTQW on a graph."""
    A = nx.adjacency_matrix(G).toarray().astype(float)
    N = len(G.nodes)
    times = [0, 1, 3, 5]
    probs = continuous_quantum_walk(A, start_node, times)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    pos = nx.spring_layout(G, seed=42)
    
    for idx, t in enumerate(times):
        ax = axes[idx]
        node_colors = probs[t]
        node_sizes = 300 + 2000 * node_colors
        
        nx.draw(G, pos, ax=ax, with_labels=True, node_color=node_colors,
                node_size=node_sizes, cmap=plt.cm.Reds, vmin=0, vmax=max(0.5, max(node_colors)),
                edge_color='gray', font_size=8)
        ax.set_title(f't = {t}', fontsize=12)
    
    plt.suptitle(f'CTQW on {name}', fontsize=14)
    plt.tight_layout()
    plt.show()

# Complete graph K_6
G_complete = nx.complete_graph(6)
plot_graph_walk(G_complete, 'Complete Graph K_6')

# Star graph
G_star = nx.star_graph(8)
plot_graph_walk(G_star, 'Star Graph S_8')

# Petersen graph
G_petersen = nx.petersen_graph()
plot_graph_walk(G_petersen, 'Petersen Graph')

## 7. Quadratic Speedup: Spreading Analysis

### 7.1 Spreading Rate Comparison

The fundamental difference between classical and quantum walks:

| Walk Type | Spreading Rate | Standard Deviation after $t$ steps |
|-----------|---------------|-----------------------------------|
| Classical random walk | Diffusive | $\sigma_{\text{cl}} = \sqrt{t}$ |
| Quantum walk | Ballistic | $\sigma_{\text{qw}} \sim t / \sqrt{2}$ |

This gives a **quadratic speedup** in spreading: to achieve standard deviation $\sigma$, the classical walk needs $t_{\text{cl}} = \sigma^2$ steps, while the quantum walk needs only $t_{\text{qw}} = \sqrt{2} \sigma$ steps.

### 7.2 Analytical Result

For the Hadamard walk with symmetric initial coin state $|\psi_C\rangle = \frac{1}{\sqrt{2}}(|0\rangle - i|1\rangle)$:

$$\sigma_{\text{qw}}(t) = \frac{t}{\sqrt{2}} \sqrt{1 - \frac{1}{\sqrt{2}}} \approx 0.54 \cdot t$$

as $t \to \infty$.

In [ ]:
# Quantitative comparison: spreading rate
times_compare = list(range(5, 101, 5))
sigma_qw = []
sigma_cl = []

for t in times_compare:
    # Quantum walk (balanced coin for symmetry)
    pos, prob = discrete_quantum_walk_1d(t, coin='balanced', initial_coin='zero')
    mean = np.sum(pos * prob)
    sigma = np.sqrt(np.sum(pos**2 * prob) - mean**2)
    sigma_qw.append(sigma)
    
    # Classical walk
    sigma_cl.append(np.sqrt(t))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(times_compare, sigma_qw, 'bo-', label='Quantum walk σ', markersize=5)
ax1.plot(times_compare, sigma_cl, 'rs-', label='Classical walk σ = √t', markersize=5)
ax1.plot(times_compare, [t/np.sqrt(2) for t in times_compare], 'b--', alpha=0.5, label='t/√2')
ax1.set_xlabel('Time steps (t)', fontsize=12)
ax1.set_ylabel('Standard deviation σ', fontsize=12)
ax1.set_title('Spreading Rate: Quantum vs Classical', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Ratio
ratios = [sq / sc for sq, sc in zip(sigma_qw, sigma_cl)]
ax2.plot(times_compare, ratios, 'go-', markersize=5, linewidth=2)
ax2.set_xlabel('Time steps (t)', fontsize=12)
ax2.set_ylabel('σ_quantum / σ_classical', fontsize=12)
ax2.set_title('Quantum Speedup Factor', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=np.sqrt(times_compare[-1])/np.sqrt(2), color='r', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At t=100: σ_quantum/σ_classical = {ratios[-1]:.2f}")
print(f"The ratio grows as √t, confirming quadratic speedup in spreading.")

## 8. Application: Quantum Walk Search

### 8.1 Grover's Search via Quantum Walks

Quantum walks provide an alternative framework for search algorithms. On a complete graph $K_N$, a quantum walk search finds a marked vertex in:

$$t_{\text{search}} = O(\sqrt{N})$$

matching Grover's speedup.

### 8.2 The Algorithm

The quantum walk search uses a modified Hamiltonian:

$$H = -\gamma A - |w\rangle\langle w|$$

where $A$ is the adjacency matrix, $\gamma$ is the hopping rate, and $|w\rangle$ is the marked vertex. The oracle adds a phase to the marked vertex, analogous to Grover's oracle.

### 8.3 Optimal Hopping Rate

For $K_N$, the optimal hopping rate is $\gamma^* = 1/N$, giving:

$$P(w, t^*) \approx 1 \quad \text{at} \quad t^* = \frac{\pi}{2}\sqrt{N}$$

In [ ]:
def quantum_walk_search(N, marked_vertex, gamma=None):
    """
    Simulate quantum walk search on complete graph K_N.
    
    Parameters:
        N: number of nodes
        marked_vertex: vertex to find
        gamma: hopping rate (default: 1/N, optimal)
    """
    if gamma is None:
        gamma = 1.0 / N
    
    # Adjacency matrix of complete graph
    A = np.ones((N, N)) - np.eye(N)
    
    # Oracle: adds energy penalty at marked vertex
    oracle = np.zeros((N, N))
    oracle[marked_vertex, marked_vertex] = 1
    
    # Hamiltonian
    H = -gamma * A - oracle
    
    # Initial state: uniform superposition
    psi0 = np.ones(N, dtype=complex) / np.sqrt(N)
    
    # Evolve over time
    t_max = 2 * np.pi * np.sqrt(N)
    times = np.linspace(0, t_max, 500)
    prob_marked = []
    
    for t in times:
        U = expm(-1j * H * t)
        psi_t = U @ psi0
        prob_marked.append(np.abs(psi_t[marked_vertex])**2)
    
    return times, prob_marked

# Search on different graph sizes
fig, ax = plt.subplots(figsize=(12, 6))

optimal_times = []
for N in [16, 32, 64, 128]:
    times, probs = quantum_walk_search(N, marked_vertex=0)
    ax.plot(times, probs, linewidth=2, label=f'N = {N}')
    
    # Find first peak
    peak_idx = np.argmax(probs)
    optimal_times.append((N, times[peak_idx], probs[peak_idx]))

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Probability of marked vertex', fontsize=12)
ax.set_title('Quantum Walk Search on Complete Graphs', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verify sqrt(N) scaling
print("\nOptimal search time scaling:")
print(f"{'N':>6} {'t_opt':>10} {'π√N/2':>10} {'P_max':>10}")
for N, t_opt, p_max in optimal_times:
    print(f"{N:6d} {t_opt:10.3f} {np.pi*np.sqrt(N)/2:10.3f} {p_max:10.4f}")

## 9. Quantum Walk Implementation in Qiskit

Let us implement a discrete-time quantum walk on a small line using actual quantum circuits.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

def build_dtqw_circuit(n_steps, n_position_qubits=4):
    """
    Build a discrete-time quantum walk circuit on a line.
    
    Uses:
    - 1 coin qubit
    - n_position_qubits for position (2^n positions)
    
    The shift operator is implemented as:
    - If coin=0: decrement position (move left)
    - If coin=1: increment position (move right)
    """
    n_pos = n_position_qubits
    coin = QuantumRegister(1, 'coin')
    position = QuantumRegister(n_pos, 'pos')
    c_pos = ClassicalRegister(n_pos, 'c_pos')
    
    qc = QuantumCircuit(coin, position, c_pos)
    
    # Start at center position (binary: 1000 for 4 qubits = position 8)
    qc.x(position[n_pos - 1])  # Set MSB to 1
    
    for step in range(n_steps):
        # Coin flip: Hadamard on coin qubit
        qc.h(coin[0])
        
        # Conditional increment (coin=1 -> move right)
        # Increment is a series of controlled operations
        for i in range(n_pos - 1, 0, -1):
            # Multi-controlled X: flip bit i if all lower bits and coin are 1
            controls = [coin[0]] + [position[j] for j in range(i)]
            qc.mcx(controls, position[i])
        qc.cx(coin[0], position[0])
        
        # Conditional decrement (coin=0 -> move left)
        qc.x(coin[0])  # Flip coin to condition on |0>
        
        # Decrement: flip bits with X first, increment, flip back
        for q in position:
            qc.x(q)
        for i in range(n_pos - 1, 0, -1):
            controls = [coin[0]] + [position[j] for j in range(i)]
            qc.mcx(controls, position[i])
        qc.cx(coin[0], position[0])
        for q in position:
            qc.x(q)
        
        qc.x(coin[0])  # Restore coin
        qc.barrier()
    
    # Measure position
    qc.measure(position, c_pos)
    
    return qc

# Build and run a small quantum walk
n_steps = 4
n_pos_qubits = 4
qc_walk = build_dtqw_circuit(n_steps, n_pos_qubits)

print(f"Quantum walk circuit: {n_steps} steps, {n_pos_qubits} position qubits")
print(f"Circuit depth: {qc_walk.depth()}")
print(f"Gate count: {sum(qc_walk.count_ops().values())}")

# Simulate
simulator = AerSimulator()
job = simulator.run(qc_walk, shots=10000)
counts = job.result().get_counts()

# Parse results
center = 2**(n_pos_qubits - 1)
positions_measured = {}
for bitstring, count in counts.items():
    pos = int(bitstring, 2) - center
    positions_measured[pos] = positions_measured.get(pos, 0) + count

# Plot
pos_sorted = sorted(positions_measured.keys())
probs_sorted = [positions_measured[p] / 10000 for p in pos_sorted]

plt.figure(figsize=(10, 5))
plt.bar(pos_sorted, probs_sorted, color='steelblue', alpha=0.8)
plt.xlabel('Position', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Quantum Walk on Qiskit ({n_steps} steps)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Applications of Quantum Walks

### 10.1 Element Distinctness

Given a function $f: [N] \to [M]$, determine if there exist $i \neq j$ with $f(i) = f(j)$. 
- Classical: $O(N)$ queries
- Quantum walk: $O(N^{2/3})$ queries (Ambainis, 2007)

### 10.2 Triangle Finding

Given a graph, determine if it contains a triangle.
- Classical: $O(N^2)$ queries to adjacency matrix  
- Quantum walk: $O(N^{5/4})$ queries

### 10.3 Spatial Search

Search on a $d$-dimensional lattice of $N$ sites:
- $d \geq 5$: $O(\sqrt{N})$ (optimal)
- $d = 4$: $O(\sqrt{N} \log N)$ 
- $d = 3$: $O(N^{1/3} \sqrt{\log N})$
- $d = 2$: $O(\sqrt{N \log N})$ (Tulsi, 2008)

### 10.4 Summary of Quantum Walk Speedups

| Problem | Classical | Quantum Walk |
|---------|-----------|-------------|
| Search (unstructured) | $O(N)$ | $O(\sqrt{N})$ |
| Element distinctness | $O(N)$ | $O(N^{2/3})$ |
| Triangle finding | $O(N^2)$ | $O(N^{5/4})$ |
| Graph connectivity | $O(N)$ | $O(N^{1/2})$ |
| Hitting time (hypercube) | $O(2^n)$ | $O(n)$ |

---

## Summary

| Concept | Key Insight |
|---------|------------|
| Classical random walk | Diffusive spreading: $\sigma \sim \sqrt{t}$ |
| DTQW | Coin + Shift operators, ballistic spreading: $\sigma \sim t$ |
| CTQW | $e^{-iAt}$ evolution, no coin needed |
| Spreading speedup | Quadratic: $O(\sqrt{t})$ classical time for $\sigma$ vs $O(t)$ quantum |
| Search applications | $O(\sqrt{N})$ search, $O(N^{2/3})$ element distinctness |

**Next:** Notebook 13 explores **Adiabatic Quantum Computing**, a fundamentally different model of quantum computation.

---

*Notebook 12 of the Quantum Computing from Ground Up series.*